# Unit 3 - Part 2 : Causal Masking, Dropout & Multi-Head
**Learn with Adi - Build an LLM from Scratch**

Executable twin of [Unit 3 - Part 2](https://aditya-402.github.io/learn-with-adi/series/llm-from-scratch/ch03b.html).
Run top to bottom; outputs match the page and the reference material (Raschka, *Build a Large Language Model (From Scratch)*).


## Setup - carried over from Part 1

In [ ]:
import torch
import torch.nn as nn

inputs = torch.tensor(
  [[0.43, 0.15, 0.89], [0.55, 0.87, 0.66], [0.57, 0.85, 0.64],
   [0.22, 0.58, 0.33], [0.77, 0.25, 0.10], [0.05, 0.80, 0.55]])
d_in, d_out = 3, 2

class SelfAttention_v2(nn.Module):
    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
    def forward(self, x):
        keys, queries, values = self.W_key(x), self.W_query(x), self.W_value(x)
        w = torch.softmax(queries @ keys.T / keys.shape[-1]**0.5, dim=-1)
        return w @ values

torch.manual_seed(789)
sa_v2 = SelfAttention_v2(d_in, d_out)

queries = sa_v2.W_query(inputs)
keys    = sa_v2.W_key(inputs)
attn_scores  = queries @ keys.T
attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
print(attn_weights)   # row 2 starts 0.2041, 0.1659, ...

## Route 1 - softmax, zero the future, renormalise
Expected row 2: `[0.5517, 0.4483, 0, 0, 0, 0]`

In [ ]:
context_length = attn_scores.shape[0]
mask_simple   = torch.tril(torch.ones(context_length, context_length))
masked_simple = attn_weights * mask_simple
row_sums      = masked_simple.sum(dim=-1, keepdim=True)
masked_simple_norm = masked_simple / row_sums
print(masked_simple_norm)

## Route 2 - mask scores with -inf, softmax once (identical result)

In [ ]:
mask   = torch.triu(torch.ones(context_length, context_length), diagonal=1)
masked = attn_scores.masked_fill(mask.bool(), -torch.inf)
print(masked)
attn_weights_causal = torch.softmax(masked / keys.shape[-1]**0.5, dim=-1)
print(attn_weights_causal)   # same as route 1

## Dropout - the compensation is visible in the numbers

In [ ]:
torch.manual_seed(123)
dropout = torch.nn.Dropout(0.5)
example = torch.ones(6, 6)
print(dropout(example))          # survivors become 2.0 = 1/(1-0.5)

torch.manual_seed(123)
print(dropout(attn_weights_causal))   # applied to the causal weights

## Batches + the CausalAttention class (stop 4)
Expected shape: `torch.Size([2, 6, 2])`

In [ ]:
batch = torch.stack((inputs, inputs), dim=0)
print(batch.shape)   # (2, 6, 3)

class CausalAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
        super().__init__()
        self.d_out   = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer('mask',
            torch.triu(torch.ones(context_length, context_length), diagonal=1))
    def forward(self, x):
        b, num_tokens, d_in = x.shape
        keys    = self.W_key(x)
        queries = self.W_query(x)
        values  = self.W_value(x)
        attn_scores = queries @ keys.transpose(1, 2)
        attn_scores.masked_fill_(self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)
        return attn_weights @ values

torch.manual_seed(123)
ca = CausalAttention(d_in, d_out, context_length=6, dropout=0.0)
context_vecs = ca(batch)
print("context_vecs.shape:", context_vecs.shape)

## Multi-head, the honest way - a wrapper
Expected first row: `[-0.4519, 0.2216, 0.4772, 0.1063]`, shape `(2, 6, 4)`

In [ ]:
class MultiHeadAttentionWrapper(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        self.heads = nn.ModuleList(
            [CausalAttention(d_in, d_out, context_length, dropout, qkv_bias)
             for _ in range(num_heads)])
    def forward(self, x):
        return torch.cat([head(x) for head in self.heads], dim=-1)

torch.manual_seed(123)
mha = MultiHeadAttentionWrapper(d_in, d_out, context_length=6, dropout=0.0, num_heads=2)
context_vecs = mha(batch)
print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

## Why the transpose matters - batched matrix multiplication

In [ ]:
a = torch.tensor([[[[0.2745, 0.6584, 0.2775, 0.8573],
                    [0.8993, 0.0390, 0.9268, 0.7388],
                    [0.7179, 0.7058, 0.9156, 0.4340]],
                   [[0.0772, 0.3565, 0.1479, 0.5331],
                    [0.4066, 0.2318, 0.4545, 0.9737],
                    [0.4606, 0.5159, 0.4220, 0.5786]]]])   # (b, heads, tokens, head_dim)

print(a @ a.transpose(2, 3))          # per-head matmul, all heads at once

first_head = a[0, 0, :, :]
print(first_head @ first_head.T)      # identical to the first block above

## MultiHeadAttention with weight splits (stop 5) - what Unit 4 imports
Expected first row: `[0.3190, 0.4858]`, shape `(2, 6, 2)`

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert (d_out % num_heads == 0), "d_out must be divisible by num_heads"
        self.d_out     = d_out
        self.num_heads = num_heads
        self.head_dim  = d_out // num_heads
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout  = nn.Dropout(dropout)
        self.register_buffer("mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1))
    def forward(self, x):
        b, num_tokens, d_in = x.shape
        keys    = self.W_key(x).view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        queries = self.W_query(x).view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        values  = self.W_value(x).view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        attn_scores = queries @ keys.transpose(2, 3)
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
        attn_scores.masked_fill_(mask_bool, -torch.inf)
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)
        context_vec = (attn_weights @ values).transpose(1, 2)
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        return self.out_proj(context_vec)

torch.manual_seed(123)
mha = MultiHeadAttention(d_in, d_out=2, context_length=6, dropout=0.0, num_heads=2)
context_vecs = mha(batch)
print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

**Exercises (reference book):** 3.2 - get 2-dim final outputs from the *wrapper* while keeping num_heads=2. 3.3 - configure MultiHeadAttention to GPT-2 small's size: 12 heads, 768-dim embeddings, context length 1024.